# Manifiesto OAI

Ejecuta `ListIdentifiers` en modo acotado, conserva eliminados y permite verificar el contrato del manifiesto sin escribir datasets.


In [ ]:
import inspect
import os
import time
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

import certifi
import pandas as pd
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning


In [ ]:
def get_oai_response(
    base_url,
    verify=None,
    max_retries=3,
    backoff_factor=1.0,
    min_interval=0.0,
    timeout=30.0,
):

    # Usa el bundle de certifi para evitar errores de certificado en requests
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    VERIFY_SSL = os.getenv("OAI_VERIFY_SSL", "false").lower() == "true"
    CA_BUNDLE = os.getenv("OAI_CA_BUNDLE") or certifi.where()
    requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

    verify_param = CA_BUNDLE if VERIFY_SSL else False
    if verify is not None:
        verify_param = verify

    for attempt in range(1, max_retries + 1):
        start_time = time.time()
        response = None
        error = None
        try:
            response = requests.get(base_url, verify=verify_param, timeout=timeout)
        except requests.RequestException as exc:
            error = exc
        elapsed_time = time.time() - start_time

        if min_interval > 0:
            wait_time = max(min_interval - elapsed_time, 0)
            if wait_time > 0:
                print(f"Pausando {wait_time:.2f} segundos para no saturar el servidor")
                time.sleep(wait_time)

        if error:
            print(f"Error en request (intento {attempt}/{max_retries}): {error}")

        if response is not None and response.status_code == 200:
            return response

        status = response.status_code if response is not None else "sin respuesta"
        print(f"Error: {status} (intento {attempt}/{max_retries})")

        if attempt < max_retries:
            backoff = backoff_factor * attempt
            print(f"Reintentando en {backoff:.2f} segundos...")
            time.sleep(backoff)
    return None


In [ ]:
def log_oai_progress(token_elem, total_processed: int):
    """Muestra el avance usando completeListSize y los registros acumulados."""
    if token_elem is None:
        return
    total = token_elem.get('completeListSize')
    try:
        total_int = int(total) if total is not None else None
        if total_int is not None and total_processed is not None:
            remaining = total_int - total_processed
            print(f"Progreso OAI: {total_processed}/{total_int} (faltan ~{remaining})")
    except ValueError:
        # Si el servidor devuelve valores no numéricos, ignora el progreso.
        pass


In [ ]:
def oai_extract_identifiers(
    base_url: str,
    context: str,
    env: str,
    source_key: str,
    repository_identifier: str,
    institution_ror: str,
    metadata_prefix: str = "oai_dc",
    dev_page_limit: int = 2,
    initial_resumption_token: str | None = None,
    page_limit: int | None = None,
    date_windows: list[dict[str, str]] | None = None,
    verify=None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build a manifest, optionally harvesting independent date windows."""
    identifiers = []
    iteration_limit = dev_page_limit if env == "dev" else page_limit
    windows = date_windows or [{}]
    if initial_resumption_token and len(windows) > 1:
        raise ValueError(
            "initial_resumption_token cannot be combined with multiple date windows"
        )

    for window in windows:
        unknown_keys = set(window) - {"from", "until"}
        if unknown_keys:
            raise ValueError(f"Unsupported OAI date window keys: {unknown_keys}")
        resumption_token = initial_resumption_token
        iteration_count = 0
        window_processed = 0

        while iteration_limit is None or iteration_count < iteration_limit:
            if resumption_token:
                query = {
                    "verb": "ListIdentifiers",
                    "resumptionToken": resumption_token,
                }
            else:
                query = {
                    "verb": "ListIdentifiers",
                    "metadataPrefix": metadata_prefix,
                    **window,
                }
            url = f"{base_url.rstrip('/')}/{context}?{urlencode(query)}"
            print(f"Consultando: {url}")

            response = get_oai_response(url, verify=verify)
            if response is None or not response.ok:
                raise RuntimeError(f"No se pudo completar el manifiesto OAI: {url}")

            try:
                root = ET.fromstring(response.text)
            except ET.ParseError as error:
                raise RuntimeError(f"Respuesta XML inválida para: {url}") from error

            ns = {"oai": "http://www.openarchives.org/OAI/2.0/"}
            headers = root.findall(".//oai:header", ns)
            for header in headers:
                identifier = header.find("oai:identifier", ns)
                datestamp = header.find("oai:datestamp", ns)
                identifiers.append(
                    {
                        "record_id": (
                            identifier.text if identifier is not None else None
                        ),
                        "datestamp": datestamp.text if datestamp is not None else None,
                        "set_id": [
                            node.text for node in header.findall("oai:setSpec", ns)
                        ],
                        "is_deleted": header.get("status") == "deleted",
                    }
                )

            iteration_count += 1
            window_processed += len(headers)
            token_elem = root.find(".//oai:resumptionToken", ns)
            resumption_token = token_elem.text if token_elem is not None else None
            log_oai_progress(token_elem, window_processed)
            if not resumption_token:
                break

    manifest = (
        pd.DataFrame(
            identifiers,
            columns=["record_id", "datestamp", "set_id", "is_deleted"],
        )
        .drop_duplicates(subset=["record_id"], keep="last")
        .reset_index(drop=True)
    )
    timestamp = pd.Timestamp.now(tz="UTC")
    manifest["_extract_datetime"] = timestamp
    manifest["_context"] = context
    manifest["_source_key"] = source_key
    manifest["_repository_identifier"] = repository_identifier
    manifest["_institution_ror"] = institution_ror
    manifest["_base_url"] = base_url.rstrip("/")
    manifest["_metadata_prefix"] = metadata_prefix
    return manifest, manifest.head(100)


In [ ]:
options = catalog.load("params:oai_extract_options").copy()
options["env"] = "dev"
if options.get("date_windows"):
    options["date_windows"] = options["date_windows"][:1]
options


In [ ]:
call_options = {name: options[name] for name in inspect.signature(oai_extract_identifiers).parameters if name in options}
df_identifiers, df_identifiers_preview = oai_extract_identifiers(**call_options)
assert df_identifiers["record_id"].notna().all()
assert not df_identifiers["record_id"].duplicated().any()
assert df_identifiers["_source_key"].notna().all()
{"rows": len(df_identifiers), "deleted": int(df_identifiers["is_deleted"].fillna(False).sum()), "sources": df_identifiers["_source_key"].value_counts().to_dict()}


In [ ]:
df_identifiers_preview
